# Integration Fundamentals

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.06 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/07_integration_fundamentals.ipynb)

---

## 🎯 Learning Objective

Connect the definite integral to area accumulation via Riemann sums, and state the fundamental theorem of calculus linking integration and differentiation.

---

## 📐 Theory

Differentiation measures instantaneous change; integration reverses direction and asks: given
a rate of change, how much accumulates in total? The two turn out to be inverse operations —
that fact alone is the deepest result in this notebook.

### Riemann sums

The **definite integral** $\int_a^b f(x)\,dx$ is defined as the area under $f$ between $a$ and
$b$, built up as a limit of rectangle approximations. Partition $[a,b]$ into $n$ strips of
width $\Delta x = (b-a)/n$, and sum the area of each rectangle:

$$\int_a^b f(x)\,dx = \lim_{n\to\infty}\sum_{i=1}^n f(x_i^*)\,\Delta x$$

where $x_i^*$ is any point in the $i$-th strip. As $n\to\infty$ and $\Delta x\to 0$, the jagged
rectangle approximation converges to the exact area — positive where $f>0$, negative where
$f<0$ (integration counts *signed* area).

### The Fundamental Theorem of Calculus

This is the result that makes integration computable without summing infinitely many
rectangles. Define $F(x) = \int_a^x f(t)\,dt$ (the "accumulated area so far" function). The
theorem states:

$$F'(x) = f(x) \qquad\text{and}\qquad \int_a^b f(x)\,dx = F(b) - F(a)$$

Part one says accumulation and differentiation *undo each other*: the derivative of "area
accumulated up to $x$" is just the height of the function at $x$. Part two says you can compute
any definite integral by finding *any* antiderivative $F$ (a function whose derivative is $f$)
and evaluating $F(b)-F(a)$ — no limits of sums required. An **indefinite integral**
$\int f(x)\,dx = F(x) + C$ is the family of all antiderivatives (the constant $C$ reflects that
shifting a function vertically doesn't change its slope).

### Basic integration rules

Integration inherits the reverse of every differentiation rule from `02.02`:
$\int x^n\,dx = \frac{x^{n+1}}{n+1}+C$ (for $n\ne -1$, the reverse power rule), integrals
distribute over sums, and constants pull outside: $\int c\,f(x)\,dx = c\int f(x)\,dx$. There's
no general product or quotient rule for integrals (unlike differentiation) — composite
integrals often need substitution or the techniques covered in `02.08`.

### Why integration matters here

An integral is fundamentally an **accumulation** or **averaging** operation, and both ideas
recur constantly once probability enters the picture (Module `03`): a probability density's
total area must equal $1$ (it's a normalized accumulation), and an expected value
$\mathbb{E}[X] = \int x\, p(x)\,dx$ is literally a weighted integral. Anywhere you see
"area under the curve," "cumulative," or "expected value" in the notebooks ahead, an integral
is doing the work underneath.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Rectangle approximations of increasing fineness converge visually to the exact area under a
curve, making the Riemann sum limit concrete rather than abstract.

In [ ]:
# Riemann sums with increasing n converging to the exact area under f(x) = sin(x) + 1.5 on [0, pi]
f = lambda x: np.sin(x) + 1.5
a, b = 0, np.pi
exact_area = -np.cos(b) + np.cos(a) + 1.5 * (b - a)  # antiderivative: -cos(x) + 1.5x

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, n in zip(axes, [4, 16, 64]):
    x_fine = np.linspace(a, b, 400)
    ax.plot(x_fine, f(x_fine), color="#4C72B0", lw=2)
    edges = np.linspace(a, b, n + 1)
    width = (b - a) / n
    midpoints = (edges[:-1] + edges[1:]) / 2
    heights = f(midpoints)
    riemann_sum = np.sum(heights * width)
    ax.bar(midpoints, heights, width=width, alpha=0.4, edgecolor="#DD8452", color="#DD8452")
    ax.set_title(f"n={n} rectangles\nsum={riemann_sum:.4f} (exact={exact_area:.4f})")
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
plt.tight_layout()
plt.show()

print(f"Exact integral (fundamental theorem): {exact_area:.6f}")
for n in [4, 16, 64, 256, 1024]:
    edges = np.linspace(a, b, n + 1)
    midpoints = (edges[:-1] + edges[1:]) / 2
    riemann_sum = np.sum(f(midpoints) * (b - a) / n)
    print(f"n={n:5d}: Riemann sum = {riemann_sum:.6f}  (error = {abs(riemann_sum - exact_area):.2e})")

## 🐍 Implementation from Scratch

We implement the trapezoidal rule (a refinement of the Riemann sum idea that connects sample
points with straight lines instead of flat rectangle tops, converging faster), then verify the
Fundamental Theorem of Calculus itself: numerically differentiate an accumulated-area function
and confirm it reproduces the original integrand.

In [ ]:
def trapezoidal_rule(f, a, b, n=1000):
    """Trapezoidal Riemann sum: average the LEFT and RIGHT edge heights of each strip
    instead of using one sample point -- this cancels out first-order error and converges faster."""
    x = np.linspace(a, b, n + 1)
    y = f(x)
    widths = np.diff(x)
    return np.sum(widths * (y[:-1] + y[1:]) / 2)

f = lambda x: np.sin(x) + 1.5
approx = trapezoidal_rule(f, 0, np.pi, n=1000)
exact = -np.cos(np.pi) + np.cos(0) + 1.5 * np.pi
print(f"Trapezoidal approximation: {approx:.6f}")
print(f"Exact (fundamental theorem): {exact:.6f}")
print(f"Cross-check with SymPy: {float(sp.integrate(sp.sin(sp.Symbol('x')) + 1.5, (sp.Symbol('x'), 0, sp.pi))):.6f}")

# --- Verify the Fundamental Theorem of Calculus itself: F'(x) should equal f(x) ---
def accumulated_area(f, a, x_val, n=2000):
    # F(x) = integral from a to x -- build it as a callable "area so far" function
    return trapezoidal_rule(f, a, x_val, n=n)

def numeric_derivative(F, x, h=1e-4):
    return (F(x + h) - F(x - h)) / (2 * h)

test_points = [0.5, 1.5, 2.5]
print("\nFundamental Theorem check: is F'(x) == f(x)?")
for x_val in test_points:
    F_prime = numeric_derivative(lambda t: accumulated_area(f, 0, t), x_val)
    print(f"  x={x_val}: F'(x) = {F_prime:.5f},  f(x) = {f(x_val):.5f},  match = {np.isclose(F_prime, f(x_val), atol=1e-3)}")

## 🤖 Why This Matters for AI

Every probability density function (Module `03.02` onward) is defined so that
$\int_{-\infty}^{\infty} p(x)\,dx = 1$ — a normalized area. Expected value,
$\mathbb{E}[X] = \int x\,p(x)\,dx$, is a weighted integral: exactly the calculation behind
average loss over a data distribution. And a metric you've likely seen reported for any
classifier, **AUC-ROC** (Area Under the ROC Curve), is a literal Riemann-sum-style integral of
the true-positive rate against the false-positive rate as a decision threshold sweeps from $0$
to $1$ — no metaphor, an actual numerical integral computed the same way we did above. Below,
we compute AUC-ROC for a toy classifier's predictions two ways: via `sklearn`'s built-in
function, and via our own `trapezoidal_rule`, confirming they agree.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# A toy binary classifier's predicted scores vs. true labels
rng = np.random.default_rng(0)
n = 300
y_true = rng.integers(0, 2, n)
# Scores that are correlated with the true label but noisy -- a realistic imperfect classifier
y_score = y_true * 0.6 + rng.normal(scale=0.4, size=n)

fpr, tpr, thresholds = roc_curve(y_true, y_score)   # sklearn gives us the (FPR, TPR) curve points

# AUC-ROC IS the area under this curve -- compute it with our own trapezoidal_rule by
# treating (fpr, tpr) as (x, y) samples of a function, sorted by x
order = np.argsort(fpr)
fpr_sorted, tpr_sorted = fpr[order], tpr[order]
auc_manual = np.sum(np.diff(fpr_sorted) * (tpr_sorted[:-1] + tpr_sorted[1:]) / 2)  # trapezoidal, directly

auc_sklearn = roc_auc_score(y_true, y_score)
print(f"AUC-ROC via sklearn's roc_auc_score: {auc_sklearn:.5f}")
print(f"AUC-ROC via our own trapezoidal integration of the ROC curve: {auc_manual:.5f}")
print(f"Match: {np.isclose(auc_sklearn, auc_manual, atol=1e-3)}")

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(fpr, tpr, color="#4C72B0", lw=2, label=f"ROC curve (AUC={auc_sklearn:.3f})")
ax.fill_between(fpr, tpr, alpha=0.2, color="#4C72B0")
ax.plot([0, 1], [0, 1], 'k--', lw=1, label="random classifier (AUC=0.5)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("AUC-ROC is a literal integral: the shaded area under this curve")
ax.legend()
plt.show()

## 🏋️ Exercises

### Warm-up
1. Compute $\int_0^2 (3x^2 - 4x + 1)\,dx$ by hand using the fundamental theorem, then verify
   with `trapezoidal_rule` and with `sp.integrate`.

### Practice
2. Use `trapezoidal_rule` to estimate $\int_0^1 e^{-x^2}\,dx$ (this integral has no elementary
   closed form — it's related to the Gaussian error function). Compare against
   `scipy.integrate.quad`, and report how many strips $n$ you need before the two agree to 6
   decimal places.

### Challenge
3. Generate predictions from a *worse* classifier (increase the noise scale in `y_score`
   generation) and a *better* one (decrease it). Recompute AUC-ROC for both with your own
   trapezoidal integration, plot all three ROC curves together, and explain in one sentence
   why a classifier that's no better than random guessing produces an ROC curve that hugs the
   diagonal (AUC ≈ 0.5).

---

## 📚 Further Reading
- Stewart, *Calculus: Early Transcendentals*, Ch. 5 (Integrals)
- Fawcett, ["An Introduction to ROC Analysis"](https://www.sciencedirect.com/science/article/pii/S016786550500303X)

---

*Next notebook: [Multivariable Calculus](08_multivariable_calculus.ipynb)*